# nnU-Net: evaluacion sobre TEST (solo inferencia)

Usa el modelo nnU-Net YA entrenado (guardado en Drive) para predecir y evaluar sobre el split **test** (held-out), y asi tener su cifra en la misma base que el resto de la tabla final. No reentrena: es solo inferencia (~20-40 min). Cualquier GPU (T4/L4/A100) sirve.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clonar repo (token) + dependencias (protocolo + nnU-Net v2)

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
os.environ['GITHUB_TOKEN']=userdata.get('GITHUB_TOKEN')
Path('/content/git_askpass.py').write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['GITHUB_TOKEN'])\n",encoding='utf-8')
os.chmod('/content/git_askpass.py',0o700)
os.environ['GIT_ASKPASS']='/content/git_askpass.py'
os.environ['GIT_TERMINAL_PROMPT']='0'
print('ok')

In [ ]:
%cd /content
!git clone https://github.com/jesusferron/tfm-brain-tumor-segmentation.git || (cd tfm-brain-tumor-segmentation && git pull origin main)
%cd /content/tfm-brain-tumor-segmentation
!pip install -r requirements/protocol.txt
!pip install -r requirements/nnunet.txt

## 3. dataset_root -> Drive (no hace falta copiar: son 243 casos, lectura unica)

In [ ]:
from pathlib import Path
import yaml
cp=Path('configs/dataset/brats_gli_2024.yaml'); c=yaml.safe_load(cp.read_text()); c['dataset_root']='/content/drive/MyDrive/TFM-datasets'; cp.write_text(yaml.safe_dump(c,sort_keys=False))
print('dataset_root =', c['dataset_root'])

## 4. Restaurar el modelo nnU-Net entrenado desde Drive

In [ ]:
!mkdir -p /content/nnUNet_raw /content/nnUNet_preprocessed /content/nnUNet_results/Dataset725_BraTSGLI2024
!cp -r "/content/drive/MyDrive/TFM-resultados/nnunet_3dfullres/nnUNet_results/." /content/nnUNet_results/Dataset725_BraTSGLI2024/
!ls /content/nnUNet_results/Dataset725_BraTSGLI2024

In [ ]:
import os
os.environ['nnUNet_raw']='/content/nnUNet_raw'
os.environ['nnUNet_preprocessed']='/content/nnUNet_preprocessed'
os.environ['nnUNet_results']='/content/nnUNet_results'
print('env ok')

## 5. Generar imagesTs con los casos de TEST
Reejecuta el conversor apuntando `--test-split test.csv`: `imagesTs` pasa a contener el split test.

In [ ]:
!python scripts/nnunet/prepare_brats_gli_nnunet_full.py --dataset-config configs/dataset/brats_gli_2024.yaml --split-dir outputs/splits/brats_gli_2024_seed20260526 --train-split train.csv --test-split test.csv --nnunet-raw /content/nnUNet_raw --dataset-id 725 --dataset-name BraTSGLI2024 --link-mode symlink

## 6. Predecir sobre TEST

In [ ]:
!nnUNetv2_predict -i /content/nnUNet_raw/Dataset725_BraTSGLI2024/imagesTs -o /content/nnunet_pred_test -d 725 -c 3d_fullres -f 0 -tr nnUNetTrainer_250epochs -chk checkpoint_best.pth

## 7. Evaluar con el `evaluate` del TFM (test)

In [ ]:
!python -m tfm_brats.cli evaluate --dataset-config configs/dataset/brats_gli_2024.yaml --split-csv outputs/splits/brats_gli_2024_seed20260526/test.csv --predictions-dir /content/nnunet_pred_test --output-csv outputs/evaluation/nnunet_3dfullres_test_metrics.csv --output-json outputs/evaluation/nnunet_3dfullres_test_metrics_summary.json
!cat outputs/evaluation/nnunet_3dfullres_test_metrics_summary.json

## 8. Guardar en Drive

In [ ]:
import shutil, os
dst='/content/drive/MyDrive/TFM-resultados/nnunet_3dfullres'
for f in ['outputs/evaluation/nnunet_3dfullres_test_metrics.csv','outputs/evaluation/nnunet_3dfullres_test_metrics_summary.json']:
    if os.path.exists(f): shutil.copy2(f, os.path.join(dst, os.path.basename(f))); print('copiado', os.path.basename(f))